# FYP: Deep Learning for Jet Classification — Google Colab runner

Train the BDT / CNN / ParticleNet / Particle Transformer top-taggers on a **free Colab T4 GPU** (~30–50× faster than a laptop CPU), then download the results back to your Mac.

**Before you start:** `Runtime → Change runtime type → Hardware accelerator: T4 GPU`.

This notebook stores everything (data + checkpoints + plots) on **Google Drive** so that if Colab disconnects, you can just re-run and it **resumes from the last completed epoch** — no work lost.

Just run the cells top to bottom. Every cell has safe defaults, so it works even if you skip the settings cell.

In [ ]:
#@title 0. Settings (defaults already point at ramc77/fyp-jet-tagging)
GITHUB_USER = "ramc77"          # your GitHub username
REPO        = "fyp-jet-tagging" # the repo name
BRANCH      = "main"
DRIVE_DIR   = "/content/drive/MyDrive/fyp_jet_tagging"  # persistent outputs
print('repo :', f'https://github.com/{GITHUB_USER}/{REPO}')
print('drive:', DRIVE_DIR)

In [ ]:
#@title 1. Check the GPU
!nvidia-smi
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

In [ ]:
#@title 2. Mount Google Drive (for persistence across disconnects)
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE_DIR = globals().get('DRIVE_DIR', '/content/drive/MyDrive/fyp_jet_tagging')
os.makedirs(os.path.join(DRIVE_DIR, 'data'), exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, 'results'), exist_ok=True)
print('Drive ready at', DRIVE_DIR)

In [ ]:
#@title 3. Clone (or update) the GitHub repo
# Self-contained: falls back to defaults if cell 0 was not run, and uses
# IPython {var} substitution (more reliable than the shell's $var).
import os
GITHUB_USER = globals().get('GITHUB_USER', 'ramc77')
REPO        = globals().get('REPO', 'fyp-jet-tagging')
BRANCH      = globals().get('BRANCH', 'main')
URL = f'https://github.com/{GITHUB_USER}/{REPO}.git'
print('cloning', URL, 'branch', BRANCH)
%cd /content
if not os.path.isdir('/content/fyp'):
    !git clone -b {BRANCH} {URL} fyp
else:
    !cd /content/fyp && git pull
%cd /content/fyp
!ls

In [ ]:
#@title 4. Install the Colab-only extras (keeps Colab's GPU torch)
!pip install -q -r requirements-colab.txt
print('done')

In [ ]:
#@title 5. Point data/ and results/ at Drive (persistent + resumable)
# Symlink the repo's data/ and results/ onto Drive so that
# (a) the 1.6 GB dataset is downloaded only once, ever, and
# (b) checkpoints survive a Colab disconnect.
import os
DRIVE_DIR = globals().get('DRIVE_DIR', '/content/drive/MyDrive/fyp_jet_tagging')
!rm -rf /content/fyp/data /content/fyp/results
os.makedirs(os.path.join(DRIVE_DIR, 'data'), exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, 'results'), exist_ok=True)
os.symlink(os.path.join(DRIVE_DIR, 'data'),    '/content/fyp/data')
os.symlink(os.path.join(DRIVE_DIR, 'results'), '/content/fyp/results')
!ls -la /content/fyp/data /content/fyp/results

In [ ]:
#@title 6. Download the dataset to Drive (only runs if not already there)
# wget -c is resumable and more robust than urllib on Colab.
%%bash
cd /content/fyp/data
for f in train val test; do
  if [ -s "${f}.h5" ]; then
    echo "[skip] ${f}.h5 already present ($(du -h ${f}.h5 | cut -f1))"
  else
    echo "[get ] ${f}.h5"
    wget -c -q --show-progress "https://zenodo.org/records/2603256/files/${f}.h5"
  fi
done
ls -lh

In [ ]:
#@title 7a. (optional) Use the FULL 1.2M-jet dataset on GPU
# On a T4 the full dataset is feasible. Skip this cell to keep the
# default 50k subset from config.py.
import re, pathlib
cfg = pathlib.Path('/content/fyp/config.py')
s = cfg.read_text()
s = re.sub(r'USE_SUBSET = True', 'USE_SUBSET = False', s)
cfg.write_text(s)
print('USE_SUBSET set to False (full dataset).')

In [ ]:
#@title 7b. Run the headline pipeline (resumable)
# If Colab disconnects mid-run, just re-run cells 2,3,5 then this one:
# it skips finished models and resumes the in-progress one from its last
# epoch (state lives on Drive).
%cd /content/fyp
!python run_full_pipeline.py

In [ ]:
#@title 8. Run the data-efficiency + calibration study (research angle)
%cd /content/fyp
!python run_study.py
!python tools/build_calibration_table.py

In [ ]:
#@title 9. Quick peek at the results
import json, glob, os
p = '/content/fyp/results/model_comparison.json'
if os.path.exists(p):
    print(json.dumps(json.load(open(p)), indent=2))
print('\nPlots produced:')
for f in sorted(glob.glob('/content/fyp/results/plots/*.pdf')):
    print('  ', os.path.basename(f))

In [ ]:
#@title 10. Zip results and download to your Mac
# (Everything is also already on your Drive under DRIVE_DIR/results.)
import shutil, os
out = '/content/fyp_results.zip'
if os.path.exists(out):
    os.remove(out)
shutil.make_archive('/content/fyp_results', 'zip', '/content/fyp/results')
print('zip size:', round(os.path.getsize(out)/1e6, 1), 'MB')
from google.colab import files
files.download(out)

## Back on your Mac

1. Unzip `fyp_results.zip` into the project's `results/` folder:
   ```bash
   cd /Users/ramchand/Desktop/Climate/May2026/FYP_particle_physics
   unzip -o ~/Downloads/fyp_results.zip -d results/
   ```
2. Re-compile the documents so they pick up the real figures + tables:
   ```bash
   cd docs/article  && pdflatex article.tex  && pdflatex article.tex
   cd ../tutorial   && pdflatex tutorial.tex && pdflatex tutorial.tex
   cd ../thesis     && pdflatex thesis.tex   && pdflatex thesis.tex
   ```

That's it — GPU does the heavy lifting, your Mac just renders the PDFs.